# Equity value vs. cash returned, capitalized at the Baa yield (daily)

Extends [FRED graph 1629621](https://fred.stlouisfed.org/graph/?graph_id=1629621) to daily frequency and adds buybacks:

$$\text{ratio}_t = \frac{P_t}{(D_t + B_t)\,/\,\text{Baa}_t}$$

- $P_t$: market value of all US corporate equities on day $t$ (\$ millions)
- $D_t$: dividends paid over the 12 months up to $t$ (\$ millions)
- $B_t$: net buybacks over the 12 months up to $t$ (\$ millions). Net buybacks = shares bought back + shares retired for cash in takeovers − new shares issued.
- $\text{Baa}_t$: Moody's Baa corporate bond yield on day $t$ (percent)

$(D+B)/\text{Baa}$ is the price at which the cash stocks pay out would yield exactly the Baa rate. The ratio compares the market's actual price with that price. At 100, stocks pay out in cash the same percentage of their price as the corporate bond pays. Above 100, the bond pays more. Dividing through by $P$ shows the same thing: ratio = 100 × Baa ÷ (cash yield in %).

**How each input is aggregated.**

| Input | Kind | Measured as |
|---|---|---|
| $P$ | level (a value at one moment) | at day $t$ |
| $\text{Baa}$ | rate (observed at one moment) | at day $t$, the same moment as $P$ |
| $D$, $B$ | flows (amounts paid over time) | sums over the trailing 12 months |

The original FRED graph averages Baa over the year while measuring $P$ at year-end. Here both are measured at the same moment.

**Where the daily values come from.** The Fed's Financial Accounts publish $P$, $D$ and $B$ quarterly, about 10 weeks after each quarter ends. For each day, the notebook starts from the latest quarter *published by that day* (assumed to be 75 days after quarter end, an approximation of the release date), then:
- $P_t$ = the Fed's quarter-end value × the S&P 500's price change since that quarter-end
- $D_t$ = the Fed's trailing-12-month dividends × the change in SPY's trailing-12-month dividends since that quarter-end. Before SPY's dividend history begins, it is held flat.
- $B_t$ = the Fed's trailing-12-month net buybacks, held flat. No free source updates buybacks between Fed releases.
- $\text{Baa}_t$ = `DBAA`, published daily from 1986.

At each publication date the daily value equals the official one. Between publications, $P$ and Baa are current, $D$ is estimated from S&P 500 dividends, and $B$ is up to about 5 months old.

**What is left out or uncertain.**
- **Buyback coverage.** The default `BB_SCOPE = 'nonfinancial'` excludes banks' and insurers' buybacks. `'all'` uses all domestic sectors, which matches $P$'s coverage. It also includes share issuance by REITs and closed-end funds, which offsets buybacks.
- **Net, not gross.** Net buybacks are far smaller than the gross totals in news coverage, because new share issuance is subtracted.

Needs a free [FRED API key](https://fredaccount.stlouisfed.org/apikey) in Colab Secrets as `FRED`. Moody's licence prohibits redistributing `BAA`/`DBAA`, so keep outputs for personal use.

In [ ]:
%pip install -q fredapi yfinance

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from fredapi import Fred
import yfinance as yf

#FRED API key: Colab Secrets (name: FRED), or the FRED_API_KEY environment variable when running locally
try:
    from google.colab import userdata
    FRED_KEY = userdata.get('FRED')
except ImportError:
    FRED_KEY = os.environ['FRED_API_KEY']
fred = Fred(api_key=FRED_KEY)

BB_SCOPE = 'nonfinancial'   #or 'all'
LAG_DAYS = 75               #approximate days from quarter end to Fed publication
B_AVG_QUARTERS = 4          #quarters of net buybacks to average: 4 = trailing 12 months; 12 = 3-year average (smooths one-off swings)

### 1. Inputs

In [ ]:
def get_by_title(title, freq, prefer_sa=None, guess_id=None):
    #Financial Accounts IDs are opaque, so try a known ID, then search FRED by exact title
    if guess_id:
        try:
            s = fred.get_series(guess_id).dropna()
            print(f'{guess_id}: {title} ({freq})')
            return s
        except Exception:
            pass
    hits = fred.search(title)
    hits = hits[(hits['title'].str.strip().str.lower() == title.lower()) & (hits['frequency_short'] == freq)]
    if hits.empty:
        return None
    pref = hits[hits['seasonal_adjustment_short'] == prefer_sa] if prefer_sa else hits
    sid = (pref if not pref.empty else hits).index[0]
    print(f'{sid}: {title} ({freq}, {hits.loc[sid, "seasonal_adjustment_short"]})')
    return fred.get_series(sid).dropna()

def period_end(idx, freq):
    return idx.to_period(freq).end_time.normalize()

MV_TITLE  = 'All Domestic Sectors; Corporate Equities; Liability, Market Value Levels'
DIV_TITLE = 'All Sectors; Dividends Paid (IMA), Transactions'
BB_TITLE  = {'nonfinancial': 'Nonfinancial Corporate Business; Corporate Equities; Liability, Transactions',
             'all':          'All Domestic Sectors; Corporate Equities; Liability, Transactions'}[BB_SCOPE]

#P: market value, quarterly level at quarter end
mv = get_by_title(MV_TITLE, 'Q', guess_id='BOGZ1LM893064105Q')
P_q = pd.Series(mv.values, period_end(mv.index, 'Q'))

#D: dividends; quarterly at an annual rate if FRED has it, otherwise the annual series the original graph uses
div_q = get_by_title(DIV_TITLE, 'Q', prefer_sa='SAAR')
if div_q is not None:
    D_ttm = pd.Series(div_q.rolling(4).mean().values, period_end(div_q.index, 'Q')).dropna()   #mean of 4 annualized quarters = 12-month flow
else:
    div_a = fred.get_series('ASDIPAA027N').dropna()
    print('ASDIPAA027N: dividends, annual (no quarterly version found; dividends anchor once a year)')
    D_ttm = pd.Series(div_a.values, period_end(div_a.index, 'Y'))

#B: net buybacks = minus net issuance, quarterly at an annual rate
iss = get_by_title(BB_TITLE, 'Q', prefer_sa='SAAR')
if iss is None:
    raise ValueError(f'No quarterly series titled {BB_TITLE!r}; look up its ID on FRED')
B_ttm = pd.Series((-iss).rolling(B_AVG_QUARTERS).mean().values, period_end(iss.index, 'Q')).dropna()

#Baa: daily from 1986, monthly before that
dbaa  = fred.get_series('DBAA').dropna()
baa_m = fred.get_series('BAA').dropna()

#daily drivers: S&P 500 price, SPY trailing dividends (sum of last 4 payments)
gspc = yf.Ticker('^GSPC').history(period='max', auto_adjust=False)['Close']
spy  = yf.Ticker('SPY').history(period='max', auto_adjust=False, actions=True)
for s in (gspc, spy):
    s.index = pd.to_datetime(s.index).tz_localize(None).normalize()
pays = spy.loc[spy['Dividends'] > 0, 'Dividends']
spy_ttm_div = pays.rolling(4).sum().reindex(spy.index).ffill().dropna()
print(f'S&P 500 price from {gspc.index[0]:%Y-%m-%d}; SPY payments: {len(pays)}, '
      f'median days between them {pays.index.to_series().diff().dt.days.median():.0f} (expect ~91)')

### 2. Official (published) values at each quarter end

Uses only the Fed's numbers and the Baa yield at quarter end. FRED's original graph has the same structure, so this also prints its annual dividends-only version for checking against the screenshot. Expect about 100 around 1958 and about 200 in the latest year. The annual check uses the Baa annual average, as FRED does.

In [ ]:
def baa_at(dates):
    #Baa at a moment: last daily DBAA reading on or before the date; before 1986, that month's average
    dates = pd.DatetimeIndex(dates)
    out = dbaa.reindex(dates, method='ffill')
    monthly = baa_m.reindex(dates.to_period('M').to_timestamp(), method=None).values
    use_monthly = dates < dbaa.index[0]
    out[use_monthly] = monthly[use_monthly]
    return out

q = pd.DataFrame({'P': P_q, 'D': D_ttm.reindex(P_q.index, method='ffill' if div_q is None else None), 'B': B_ttm}).dropna()
q['baa'] = baa_at(q.index).values
q['ratio_div'] = q['P'] / (q['D'] / q['baa'])
q['ratio'] = (q['P'] / ((q['D'] + q['B']) / q['baa'])).where(q['D'] + q['B'] > 0)

#check against the original FRED graph (annual market value, annual dividends, annual-average Baa)
mv_a = get_by_title(MV_TITLE, 'A', guess_id='BOGZ1LM893064105A')
chk = pd.DataFrame({'mv': mv_a, 'div': fred.get_series('ASDIPAA027N')}).dropna()
chk.index = chk.index.year
chk['fred_graph'] = chk['mv'] / (chk['div'] / baa_m.groupby(baa_m.index.year).mean())
print(chk['fred_graph'].loc[[y for y in (1950, 1958, 2000) if y in chk.index] + [chk.index[-1]]].round(1))
print()
print(q[['P', 'D', 'B', 'baa', 'ratio_div', 'ratio']].iloc[::20].round(1).to_string())

### 3. Daily values

In [ ]:
def as_of(anchor, driver, start, end):
    #For each day: latest anchor value published by then, scaled by the driver's change since the anchor date
    days = pd.bdate_range(start, end)
    a = pd.DataFrame({'value': anchor.values, 'anchor_date': anchor.index,
                      'available': anchor.index + pd.Timedelta(days=LAG_DAYS)}).sort_values('available')
    out = pd.merge_asof(pd.DataFrame({'date': days}), a, left_on='date', right_on='available').set_index('date')
    if driver is not None:
        drv_now = driver.reindex(days, method='ffill')
        drv_then = driver.reindex(out['anchor_date'], method='ffill').values
        growth = (drv_now.values / drv_then)
        growth[pd.isna(growth)] = 1.0     #no driver data (e.g. before SPY dividends begin): hold the anchor flat
        out['value'] = out['value'] * growth
    return out[['value', 'anchor_date']]

start = max(dbaa.index[0], gspc.index[0], q.index[0] + pd.Timedelta(days=LAG_DAYS))
end = min(dbaa.index[-1], gspc.index[-1])
P = as_of(q['P'], gspc, start, end)
D = as_of(D_ttm, spy_ttm_div, start, end)     #anchored on the dividend series' own dates, so growth since its latest value is added
B = as_of(q['B'], None, start, end)

daily = pd.DataFrame({'P': P['value'], 'D': D['value'], 'B': B['value'],
                      'baa': dbaa.reindex(P.index, method='ffill', limit=5),
                      'P_from': P['anchor_date'], 'D_from': D['anchor_date'], 'B_from': B['anchor_date']}).dropna()
daily = daily[daily.index.isin(gspc.index)]     #trading days only
daily['ratio_div'] = daily['P'] / (daily['D'] / daily['baa'])
daily['ratio'] = (daily['P'] / ((daily['D'] + daily['B']) / daily['baa'])).where(daily['D'] + daily['B'] > 0)
print(daily.tail().round({'P': 0, 'D': 0, 'B': 0, 'baa': 2, 'ratio_div': 1, 'ratio': 1}).to_string())

### 4. Charts

In [ ]:
BLUE, ORANGE, INK, MUTED = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK,
                     'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.grid': True, 'grid.color': '#e4e3df', 'grid.linewidth': 0.6})

def style(ax, title, years=10):
    ax.axhline(100, color=INK, ls='--', lw=1)
    ax.set_title(title, loc='left', fontsize=11, color=INK)
    ax.legend(frameon=False, loc='best')
    ax.xaxis.set_major_locator(mdates.YearLocator(years))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig, axes = plt.subplots(2, 1, figsize=(11, 9))

ax = axes[0]
ax.plot(q.index, q['ratio_div'], color=BLUE, lw=1.6, label='P / (D / Baa)')
ax.plot(q.index, q['ratio'], color=ORANGE, lw=1.6, label='P / ((D + B) / Baa)')
style(ax, f'Quarterly, published Fed data   (buybacks: {BB_SCOPE})')

ax = axes[1]
ax.plot(daily.index, daily['ratio_div'], color=BLUE, lw=1.1, label='P / (D / Baa)')
ax.plot(daily.index, daily['ratio'], color=ORANGE, lw=1.1, label='P / ((D + B) / Baa)')
ax.plot(q.index, q['ratio'], 'o', ms=3, color=INK, label='Official quarter-end values (with buybacks)')
ax.set_xlim(daily.index[0], daily.index[-1])
style(ax, f'Daily   (latest {daily.index[-1]:%Y-%m-%d}: {daily.ratio_div.iloc[-1]:.0f} dividends only, '
          f'{daily.ratio.iloc[-1]:.0f} with buybacks)', years=5)
fig.tight_layout()
plt.show()

In [ ]:
recent = daily.loc[daily.index[-1] - pd.DateOffset(years=2):]
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(recent.index, recent['ratio'], color=ORANGE, lw=1.5, label='P / ((D + B) / Baa)')
ax.plot(recent.index, recent['ratio_div'], color=BLUE, lw=1.5, label='P / (D / Baa)')
style(ax, 'Daily, last 2 years', years=1)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
fig.tight_layout()
plt.show()

last = daily.iloc[-1]
print(f'Latest ({daily.index[-1]:%Y-%m-%d})')
print(f'  P   = ${last.P/1e6:,.1f} trillion (Fed {last.P_from:%Y-%m-%d}, moved by S&P 500 since)')
print(f'  D   = ${last.D/1e6:,.2f} trillion / yr (Fed {last.D_from:%Y-%m-%d}, moved by SPY dividends since)')
print(f'  B   = ${last.B/1e6:,.2f} trillion / yr (Fed {last.B_from:%Y-%m-%d}, held flat, {B_AVG_QUARTERS}-quarter average)')
print(f'  Baa = {last.baa:.2f}%')
print(f'  cash yield (D+B)/P = {100*(last.D+last.B)/last.P:.2f}%  ->  ratio {last.ratio:.0f}')

### 5. Speculation measure and the decline that would signal a pop

Price model: $P = (D + B)/r$ with $r$ = Baa, so

$$\text{speculation} = \frac{P}{(D + B)/r}$$

Holding $D$, $B$ and $r$ at today's values, the measure moves one-for-one with price. The S&P 500 level that brings it back to a target $T$ is therefore today's level × $T$ ÷ today's reading. The table shows that level and the fall it implies from the S&P 500's peak.

In [ ]:
spec_d, spec_q = daily['ratio'], q['ratio'].dropna()     #with buybacks
now = spec_d.iloc[-1]
spx_now, spx_peak, peak_date = gspc.iloc[-1], gspc.max(), gspc.idxmax()

targets = {
    'No speculation (P = (D+B)/r)': 100,
    'Median since 1952 (quarterly)': spec_q.median(),
    'Median since 1986 (daily)': spec_d.median(),
    '75th percentile since 1952': spec_q.quantile(0.75),
}
rows = []
for name, t in targets.items():
    spx_t = spx_now * t / now
    rows.append({'target': name, 'speculation level': round(t), 'S&P 500 level': round(spx_t),
                 'fall from today': f'{1 - spx_t / spx_now:.0%}', 'fall from peak': f'{1 - spx_t / spx_peak:.0%}'})
print(f'Speculation now: {now:.0f}   S&P 500 now: {spx_now:,.0f}   peak: {spx_peak:,.0f} on {peak_date:%Y-%m-%d}')
print(pd.DataFrame(rows).set_index('target').to_string())

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(spec_d.index, spec_d, color=ORANGE, lw=1.2, label='P / ((D + B) / r)')
for name, t in list(targets.items())[1:3]:
    ax.axhline(t, color=MUTED, lw=0.8, ls=':')
    ax.text(spec_d.index[0], t, f' {name}', color=MUTED, fontsize=8, va='bottom')
style(ax, f'Speculation, daily   (latest {spec_d.index[-1]:%Y-%m-%d}: {now:.0f})', years=5)
fig.tight_layout()
plt.show()